This will be a more detailed implementation of the Black-Scholes Equation in European Options, taking into account other variables.

In [4]:
#Import dependencies
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from scipy.stats import norm
from fredapi import Fred
import creds

In [2]:
#Probability Density function; phi(d)
N = norm.cdf

#Define Call Option
def BlackSchole_Call(S, X, T, r, sigma):
    d1 = (np.log(S/X) + (r + (pow(sigma, 2) / 2)) * T) / (sigma * np.sqrt(T))
    d2 = d1 - (sigma * np.sqrt(T))

    return S*N(d1) - X*np.exp(-r*T)*N(d2)

#Define Put Option
def BlackSchole_Put(S, X, T, r, sigma):
    d1 = (np.log(S/X) + (r + (pow(sigma, 2) / 2)) * T) / (sigma * np.sqrt(T))
    d2 = d1 - (sigma * np.sqrt(T))
    return X*np.exp(-r*T)*N(-d2) - S*N(-d1)

Parameters:

The given parameters will be better calculated using other equations to derive the values.

Risk-Free Rate of Return: a theoretical concept that an investment that guarantees returns without any risks. (on paper)
Higher risk == higher chance for the investment to lose profit.

For risk-free investment to work:
1. you have to know your expected return with certainty
2. the entity making cash flow == NO default risk
3. No reinvestment risk

I will use data from FRED to calculate the Risk-Free Interest Rate of United Kingdom (Since we are looking at European model).

*Real Risk-Free Rate: Nominal Rate (e.g. 10-Year Bond Yield) - Expected Inflaton rate.*


In [5]:
#Test of FRED Api
#Importing data of UK Bond rates
fred = Fred(creds.api_key)

#10-year bond yields
UK_bond_yield = fred.get_series('IRLTLT01GBM156N')

#UK CPI YoY
UK_inflation = fred.get_series('CPALTT01GBM657N')

print("Latest UK 10-Year Bond Yield:")
print(UK_bond_yield.tail(1))

print("\nLatest UK YoY Inflation Rate (CPI):")
print(UK_inflation.tail(1))

#Calculate Risk-Free Rate
#Align dates
latest_date = min(UK_bond_yield.dropna().index[-1],
                  UK_inflation.dropna().index[-1])

rf_rate = UK_bond_yield.loc[latest_date] - UK_inflation[latest_date]
print(f"\nEstimated UK Real risk-free rate: {rf_rate:.2f}%")

Latest UK 10-Year Bond Yield:
2025-06-01    4.5248
dtype: float64

Latest UK YoY Inflation Rate (CPI):
2024-02-01    0.6
dtype: float64

Estimated UK Real risk-free rate: 3.52%


Implied Volatility:

Since it is difficult to compute the Implied Volatility as it does not have a closed-form solution, we will take into account for that and try to estimate its initial value using Brenner & Subrahmanyam's estimate from 1988:

$\sigma \approx \sqrt{\frac{2\pi}{T} \cdot \frac{C}{S}}$

Where:
- T: Total time to expiry
- C: price of option on the stock (call / put)
- S: stock price


In [7]:
#Initial Parameters
days = 30   #No. of historical Days to analyze
T_total = days / 365 #Total time to expiration (in years)

stock = yf.Ticker("SPY")
hist = stock.history(period=f'{days+5}d')

calls_df = stock.option_chain().calls
calls_df.head(10)



,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,SPY250723C00500000,2025-07-17 17:48:18+00:00,500.0,127.51,133.26,136.06,0.000000,0.000000,NaN,1,2.191411,True,REGULAR,USD
1,SPY250723C00520000,2025-07-15 16:24:54+00:00,520.0,105.11,113.26,116.06,0.000000,0.000000,NaN,1,1.875001,True,REGULAR,USD
2,SPY250723C00550000,2025-07-22 15:04:05+00:00,550.0,77.85,83.26,86.08,0.000000,0.000000,192.0,15,1.417972,True,REGULAR,USD
3,SPY250723C00560000,2025-07-23 14:23:38+00:00,560.0,70.28,73.26,76.08,1.250000,1.810807,1.0,6,1.264652,True,REGULAR,USD
4,SPY250723C00575000,2025-07-23 18:42:11+00:00,575.0,58.27,58.26,61.08,5.970001,11.414916,22.0,6,1.036138,True,REGULAR,USD
5,SPY250723C00585000,2025-07-15 19:59:54+00:00,585.0,37.86,48.26,51.08,0.000000,0.000000,NaN,1,0.883790,True,REGULAR,USD
6,SPY250723C00590000,2025-07-23 20:11:26+00:00,590.0,43.95,43.26,46.08,6.290001,16.702074,60.0,59,0.806643,True,REGULAR,USD
7,SPY250723C00591000,2025-07-21 14:46:40+00:00,591.0,41.36,42.26,45.08,1.209999,3.013696,2.0,4,0.791506,True,REGULAR,USD
8,SPY250723C00592000,2025-07-18 18:24:15+00:00,592.0,41.34,41.26,44.08,5.779999,16.254213,10.0,2,0.775881,True,REGULAR,USD
9,SPY250723C00595000,2025-07-23 20:02:20+00:00,595.0,39.02,38.26,41.08,5.549999,16.582012,11.0,19,0.729495,True,REGULAR,USD


In [ ]:
#Analytical Method

#Parameters
r = rf_rate #risk-free interest rate; Using estimation from above
sigma = 0.30 #Constant Implied volatility

